In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Subject–Time ICA on Wavelet Power

## Scope

This notebook prepares the data and runs the ICA decomposition on the
`(F × C, S × T)` reshape of the 4-D wavelet power tensor. Frequencies
and channels form the observation axis; subjects and time are combined
into the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_freqs × n_channels,  n_subjects × n_times)
         ──── observations ─────  ──── features ──────
```

Before reshaping the tensor is **z-scored along the time axis** so that
every `(subject, channel, frequency)` slice has zero mean and unit
variance.  This removes overall amplitude differences and ensures that
PCA/ICA operates on standardised activations.

ICA components live in the `S × T` feature space, so each component is
reshaped back to `(n_subjects, n_times)` — a **subject × time pattern**
shared across frequencies and channels.

## Pipeline

1. Load wavelet power for the chosen condition / music type (cached).
2. Apply subject / channel / time subsets for fast iteration.
3. Z-score along time and reshape to `(F×C, S×T)`.
4. PCA dimensionality reduction.
5. FastICA on the PCA scores.

After the final cell the following variables are available for any
downstream analysis:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_fc` | `(F×C, S×T)` | Z-scored reshaped 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(F×C, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(F×C, K_ica)` | ICA scores (per-observation weights) |
| `ica_components` | `(K_ica, S×T)` | ICA subject-temporal component patterns |
| `scores_2d` | `(F, C, K)` | ICA scores reshaped to frequency × channel |
| `components_2d` | `(K, S, T)` | ICA components reshaped to subject × time |

## Configuration

In [ ]:
# ── Experiment configuration ────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ─────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ───────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ──────────────────────────────────────────────
N_COMPONENTS_PCA = 50  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "subject_time"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance.  This ensures that PCA/ICA are not
dominated by high-power channels, subjects, or frequency bands.

**Reshaping** combines frequencies and channels into the observation axis,
and subjects and time into the feature axis:

```
(S, C, F, T)  →  transpose to  (F, C, S, T)
              →  reshape to     (F × C,  S × T)
                                observations  features
```

Each row of the resulting 2-D matrix is the z-scored power across all
subjects and time points for a single frequency at a single channel.
PCA/ICA will discover **subject-temporal patterns** — subject × time
fingerprints — shared across frequencies and channels.

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape: (S, C, F, T) → transpose → (F, C, S, T) → (F*C, S*T)
bb_z_fc = bb_z.transpose(2, 1, 0, 3)  # (F, C, S, T)
n_obs = n_freqs * n_channels
n_feat = n_subjects * n_times
X_fc = bb_z_fc.reshape(n_obs, n_feat)  # (F*C, S*T)

print(f"Reshaped matrix shape : {X_fc.shape}")
print(f"  Observations (F×C)  : {X_fc.shape[0]}")
print(f"  Features     (S×T)  : {X_fc.shape[1]}")
print(f"Row means  ≈ 0 : {X_fc.mean(axis=1).mean():.6f}")
print(f"Row stds        : {X_fc.std(axis=1).mean():.4f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

We first reduce the `S × T` feature space to `N_COMPONENTS_PCA`
principal components, keeping the directions of maximum variance.  Then
FastICA rotates the PCA subspace to maximise statistical independence,
yielding `N_COMPONENTS_ICA` independent components.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_scores` | `(F×C, K)` | Per-observation weight for each IC |
| `ica_components` | `(K, S×T)` | Subject-temporal pattern of each IC |
| `scores_2d` | `(F, C, K)` | ICA scores reshaped to frequency × channel |
| `components_2d` | `(K, S, T)` | ICA components reshaped to subject × time |

In [ ]:
# --- PCA ---
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca_scores = pca.fit_transform(X_fc)  # (F*C, K_pca)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot — {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance — {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_scores = ica.fit_transform(pca_scores)  # (F*C, K_ica)
ica_components = ica.components_ @ pca.components_  # (K_ica, S*T)

# Reshape ICA scores to (F, C, K) for downstream analysis
scores_2d = ica_scores.reshape(n_freqs, n_channels, N_COMPONENTS_ICA)  # (F, C, K)

# Reshape ICA components to (K, S, T) for subject-temporal analysis
components_2d = ica_components.reshape(
    N_COMPONENTS_ICA, n_subjects, n_times
)  # (K, S, T)

print(f"ICA scores shape       : {ica_scores.shape}")
print(f"ICA components shape   : {ica_components.shape}")
print(f"Scores 2-D shape       : {scores_2d.shape}  (F, C, K)")
print(f"Components 2-D shape   : {components_2d.shape}  (K, S, T)")

---
## Analysis (a) — Intersubject Correlation Matrix

For each ICA component we compute a **subject × subject** Pearson
correlation matrix using each subject's temporal profile (one row of
`components_2d[k]`, length `T`).

High off-diagonal correlations indicate that the component captures a
consistent temporal activation pattern across individuals — a hallmark
of stimulus-driven (rather than noise-driven) modes.

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each row of components_2d[i] is one subject's temporal profile (length T);
    # np.corrcoef rows-as-variables gives the (S, S) inter-subject correlation.
    corr_mat = np.corrcoef(components_2d[i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Temporal Profiles — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Mean Subject Loading per Component

For each ICA component, compute the **mean absolute temporal activation**
per subject. Since components are shaped `(S, T)`, taking the mean of
the absolute values over time gives a scalar per `(subject, component)`
pair — a summary of how strongly each participant expresses the
subject-temporal mode across the recording.

Subjects with uniformly high loadings indicate a stimulus-driven mode;
uneven loadings reflect individual differences.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `subject_loadings` | `(S, K)` | Mean `|component|` over time, per subject and IC |

In [ ]:
# Subject loadings: mean |component value| over time
subject_loadings = np.abs(components_2d).mean(axis=2).T  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="darkorange",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|activation|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Mean Loading per Component — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Time–Frequency Map per Component (Outer Product)

For each ICA component we build a **frequency × time** map directly from
the ICA decomposition, without touching the raw z-scored power:

```
A = scores_2d        # (F, C, K)  — per-observation (freq × chan) weights
S = components_2d    # (K, S, T)  — per-component subject × time patterns

freq_profile[k]  = A[:, :, k].mean(axis=1)    # (F,)  channel-averaged score
time_profile[k]  = S[k].mean(axis=0)          # (T,)  subject-averaged activation
tf_map[k]        = outer(freq_profile[k], time_profile[k])  # (F, T)
```

The outer product gives a rank-1 approximation of the component's
frequency–time structure grounded entirely in the ICA solution.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `freq_profiles` | `(F, K)` | Channel-averaged ICA score per frequency |
| `time_profiles` | `(K, T)` | Subject-averaged IC temporal activation |
| `ft_maps` | `(K, F, T)` | Outer-product frequency × time map per component |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

# Frequency profile: collapse channels from ICA scores  (F, C, K) → (F, K)
freq_profiles = scores_2d.mean(axis=1)  # (F, K)

# Time profile: average subject activations from ICA components  (K, S, T) → (K, T)
time_profiles = components_2d.mean(axis=1)  # (K, T)

# Outer product for each component: (F, K) x (K, T) → (K, F, T)
ft_maps = np.einsum("fk,kt->kft", freq_profiles, time_profiles)  # (K, F, T)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_maps[i]  # (F, T)
    vlim_i = np.percentile(np.abs(data_i), 99)
    mesh = ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="RdBu_r",
        vmin=-vlim_i,
        vmax=vlim_i,
        shading="auto",
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} — Freq × Time Map (outer product)", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency × Time Maps per IC — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (d) — Mean Component Channel Loading (Topomap)

The spatial fingerprint of each ICA component lives in the scores
`(F, C, K)`. Averaging the ICA scores across frequencies collapses the
spectral axis and yields a per-channel loading vector for every IC:

```
A = scores_2d                                # (F, C, K)
chan_loading[:, k] = A[:, :, k].mean(axis=0) # (C,)  freq-averaged channel loading
```

Each component is plotted on its **own symmetric color scale** so that
weak components are not visually flattened by stronger ones.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `chan_loading` | `(C, K)` | Freq-averaged ICA channel score per IC |

In [ ]:
# Per-component channel loading from ICA scores, averaged over frequencies:
#   chan_loading[c, k] = mean_f  scores_2d[f, c, k]                  (C, K)
chan_loading = scores_2d.mean(axis=0)  # (C, K)

# Get MNE Info for topomap (restrict to the channel subset used here)
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Per-component symmetric color scale around zero
    vlim_i = np.percentile(np.abs(chan_loading[:, i]), 99)
    im, _ = plot_topomap(
        chan_loading[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-vlim_i, vlim_i),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f"Mean Component Channel Loading (topomap) — {LABEL}",
    fontsize=12,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (e) — Mean and Variance of IC Signal Over Time Across Subjects

Each IC activation is shaped `(S, T)`, so at every time point we
summarise the across-subject distribution with:

```
mean_temporal[k, t] = mean_s [ components_2d(k, s, t) ]    # (K, T)
var_temporal[k, t]  = var_s  [ components_2d(k, s, t) ]    # (K, T)
std_temporal[k, t]  = sqrt(var_temporal[k, t])             # (K, T)
```

The **mean** is plotted as a line; the **variance** is shown as an
error-shading band around the mean (`mean ± std`, i.e. `±√variance` —
kept in the same units as the mean for direct comparison).  Wide bands
mark idiosyncratic moments; narrow bands mark cross-subject consensus.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `mean_temporal` | `(K, T)` | Mean IC activation across subjects |
| `var_temporal` | `(K, T)` | Variance of IC activation across subjects |
| `std_temporal` | `(K, T)` | Standard deviation, used as error-band width |

In [ ]:
# Across-subject summaries of each IC's temporal activation
mean_temporal = components_2d.mean(axis=1)  # (K, T)
var_temporal = components_2d.var(axis=1)  # (K, T)
std_temporal = np.sqrt(var_temporal)  # (K, T) — used as the error-band width

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.9, color="darkorange", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="darkorange",
        label="± √variance",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} — Mean & Variance Across Subjects", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Per-IC Mean and Variance of Activation Over Time — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_mean_variance_over_time.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (f) — Sliding-Window LOO-ISC per Component

For each ICA component (shape `(S, T)`) we compute a **time-resolved
leave-one-out inter-subject correlation** by sliding a window over the
time axis. Inside each window:

```
for each subject s:
    others_mean = mean over s' ≠ s  of  components_2d[k, s', win]
    r[k, w, s]  = pearson( components_2d[k, s, win], others_mean )
```

A single `(window_sec, step_sec)` pair is used (`SLIDING_WINDOW_SEC`,
`SLIDING_STEP_SEC`) — the smallest step we considered, giving the
finest temporal resolution. The across-subject **mean** is drawn as a
stair-line with the `± √variance` band around it.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `loo_isc_sw['per_subject']` | `(K, W, S)` | Per-subject LOO-ISC per IC |
| `loo_isc_sw['mean']` | `(K, W)` | Across-subject mean LOO-ISC |
| `loo_isc_sw['std']` | `(K, W)` | Std of LOO-ISC across subjects |
| `loo_isc_sw['edges']` | `(W,)` | Window-start times in seconds |

In [ ]:
from scipy.stats import pearsonr  # noqa: E402

# Single sliding-window setting — smallest step for finest temporal resolution
SLIDING_WINDOW_SEC = 2.0
SLIDING_STEP_SEC = 0.5

t_end = n_times / sfreq  # data end time, used to close the last stair


def _compute_sliding_loo_isc(window_sec: float, step_sec: float) -> dict:
    """Per-subject and across-subject LOO-ISC for one (window, step) pair."""
    win_samples = int(round(window_sec * sfreq))
    step_samples = int(round(step_sec * sfreq))
    if win_samples < 2 or win_samples > n_times:
        raise ValueError(
            f"window_sec={window_sec}s -> {win_samples} samples; "
            f"must be in [2, {n_times}]."
        )
    if step_samples < 1:
        raise ValueError(
            f"step_sec={step_sec}s -> {step_samples} samples; must be ≥ 1."
        )

    starts = np.arange(0, n_times - win_samples + 1, step_samples)
    # Window-start times so the first stair begins at t=0.
    edges = starts / sfreq

    per_subject = np.zeros((N_COMPONENTS_ICA, len(starts), n_subjects))
    for k in range(N_COMPONENTS_ICA):
        comp = components_2d[k]  # (S, T)
        for w_idx, start in enumerate(starts):
            win = comp[:, start : start + win_samples]
            for s in range(n_subjects):
                others_mean = np.delete(win, s, axis=0).mean(axis=0)
                per_subject[k, w_idx, s] = float(pearsonr(win[s], others_mean)[0])

    return {
        "per_subject": per_subject,  # (K, W, S)
        "mean": per_subject.mean(axis=2),  # (K, W)
        "std": per_subject.std(axis=2),  # (K, W)
        "edges": edges,  # (W,) window-start times
    }


def _close_to_end(edges: np.ndarray, vals: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Append t_end and duplicate the last value so steps-post extends to t_end."""
    if edges[-1] >= t_end:
        return edges, vals
    return np.append(edges, t_end), np.append(vals, vals[-1])


loo_isc_sw = _compute_sliding_loo_isc(SLIDING_WINDOW_SEC, SLIDING_STEP_SEC)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.8 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    edges_m, mean_m = _close_to_end(loo_isc_sw["edges"], loo_isc_sw["mean"][i])
    _, std_m = _close_to_end(loo_isc_sw["edges"], loo_isc_sw["std"][i])
    ax.plot(
        edges_m,
        mean_m,
        lw=1.2,
        color="steelblue",
        drawstyle="steps-post",
        label="mean" if i == 0 else None,
    )
    ax.fill_between(
        edges_m,
        mean_m - std_m,
        mean_m + std_m,
        alpha=0.2,
        color="steelblue",
        step="post",
        label="± √variance" if i == 0 else None,
    )
    ax.axhline(0.0, ls="--", lw=0.6, color="gray")
    ax.set_xlim(0.0, t_end)
    ax.set_ylabel(f"IC {i + 1}\nLOO-ISC")
    ax.set_ylim(-1.05, 1.05)
    ax.set_title(f"Component {i + 1} — Sliding-Window LOO-ISC", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Per-IC Sliding-Window LOO-ISC "
    f"(win={SLIDING_WINDOW_SEC:.1f}s, step={SLIDING_STEP_SEC:.1f}s) — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_sliding_window_loo_isc.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (g) — Subject × Time Heatmap per Component

Each ICA component already has shape `(S, T)`, so we plot it directly
as a heatmap with **subjects on the y-axis** and **time on the x-axis**.
This is the subject-time analogue of the subject × frequency heatmap
in the time-features notebook: rows reveal each participant's temporal
fingerprint of the component, while columns reveal moments where
activation is consistent across subjects.

A diverging colormap (`RdBu_r`, symmetric around zero) preserves the
sign of the ICA component values, with limits shared across the
displayed components for cross-component comparability.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `components_2d` | `(K, S, T)` | ICA component values per (IC, subject, time) |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

# Symmetric color limits around zero, shared across the displayed components
_vlim_st = np.percentile(np.abs(components_2d[:n_show]), 99)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

mesh = None
for i, ax in enumerate(axes):
    mesh = ax.pcolormesh(
        time,
        np.arange(n_subjects),
        components_2d[i],
        cmap="RdBu_r",
        vmin=-_vlim_st,
        vmax=_vlim_st,
        shading="auto",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_ylabel("Subject")
    ax.set_title(f"IC {i + 1} — Subject × Time Activation", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025, label="activation")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Subject × Time Activation Heatmaps per IC — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_time_heatmap.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (h) — Frequency × Channel Heatmap per Component

Companion to the topomap in (d): the ICA scores `(F, C, K)` are shown
**without collapsing the frequency axis**, so each component appears as
a `frequency × channel` heatmap. The topomap is the column-average of
this map projected onto the scalp; the heatmap reveals which channels
contribute through which frequency bands.

A diverging colormap (`RdBu_r`) preserves the sign of the scores, and
each component uses its **own symmetric color scale** so weak ICs are
not flattened by stronger ones.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `scores_2d` | `(F, C, K)` | ICA scores per (frequency, channel, IC) |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

channels = np.arange(n_channels)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4.5), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = scores_2d[:, :, i]  # (F, C)
    vlim_i = np.percentile(np.abs(data_i), 99)
    mesh = ax.pcolormesh(
        channels,
        FREQS,
        data_i,
        cmap="RdBu_r",
        vmin=-vlim_i,
        vmax=vlim_i,
        shading="auto",
    )
    ax.set_xlabel("Channel")
    ax.set_title(f"IC {i + 1}", fontsize=10)
    fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="score")

axes[0].set_ylabel("Frequency (Hz)")
fig.suptitle(
    f"Frequency × Channel ICA Score Heatmaps — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_freq_channel_heatmap.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")